# NLP with Disaster Tweets: DeBERTa-v3-Large Approach
本ノートブックでは、Kaggleコンペティション「Natural Language Processing with Disaster Tweets」に向けた高精度なテキスト分類モデルを構築します。
SOTAのTransformerモデルである `microsoft/deberta-v3-large` を採用し、Optunaによるハイパーパラメータ最適化を組み込むことで、短文かつノイズの多いSNSデータから「実際の災害報」をロバストに検知するパイプラインを実装します。

## 1. Environment Setup & Data Loading
分析に必要な各種ライブラリ（PyTorch, Hugging Face Transformers等）のインポート、およびKaggle環境からのデータセット読み込みを行います。

In [ ]:
# 1. 基盤ライブラリと機械学習フレームワークのインポート
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import re
from tqdm import tqdm
import torch
import os
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from transformers import get_linear_schedule_with_warmup
import gc
import torch
import os

# GPUメモリの断片化を防ぐための環境変数設定
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. データセットのロードと確認
df_train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
df_sample = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

# 最終的な推論・提出フェーズで使用するためテストデータのIDを保持
test_ids = df_test["id"].copy()

display(df_train.head())
display(df_test.head())
display(df_sample.head())

# 本タスクにおける基盤モデル（Backbone）の指定
bert_model = "microsoft/deberta-v3-large"

## 2. Transformer Model Initialization
指定されたDeBERTa-v3-largeモデルのトークナイザーと事前学習済み重みをロードし、計算リソース（GPU）へとマッピングします。

In [ ]:
# トークナイザーと事前学習済みモデルのインスタンス化
tokenizer = AutoTokenizer.from_pretrained(bert_model)
model = AutoModel.from_pretrained(bert_model)

# 計算リソースの自動判別と割り当て（CUDA環境の優先活用）
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# モデルを計算デバイス（GPU/CPU）へ転送
model.to(device)

## 3. Data Preprocessing
SNS特有の略語、特殊記号、およびHTMLエンティティを正規化する前処理パイプラインを定義します。モデルのAttention機構が文脈を正確に捉えられるよう、テキストの品質を向上させる重要な工程です。

In [ ]:
# 1. テキスト正規化（クリーニング）関数の定義
# 英語の短縮形を正式な表現に展開するためのマッピング辞書
cList = {
  "ain't": "am not", "aren't": "are not", "can't": "cannot", "can't've": "cannot have",
  "'cause": "because", "could've": "could have", "couldn't": "could not", "couldn't've": "could not have",    "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not",
  "hadn't've": "had not have", "hasn't": "has not", "haven't": "have not", "he'd": "he would",
  "he'd've": "he would have", "he'll": "he will", "he'll've": "he will have", "he's": "he is",
  "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
  "i'd": "i would", "i'd've": "i would have", "i'll": "i will", "i'll've": "i will have",
  "i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it had", "it'd've": "it would have",
  "it'll": "it will", "it'll've": "it will have", "it's": "it is", "let's": "let us",
  "ma'am": "madam", "mayn't": "may not", "might've": "might have", "mightn't": "might not",
  "mightn't've": "might not have", "must've": "must have", "mustn't": "must not",
  "mustn't've": "must not have", "needn't": "need not", "needn't've": "need not have",
  "o'clock": "of the clock", "oughtn't": "ought not", "oughtn't've": "ought not have",
  "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have",
  "she'd": "she would", "she'd've": "she would have", "she'll": "she will",
  "she'll've": "she will have", "she's": "she is", "should've": "should have",
  "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have",
  "so's": "so is", "that'd": "that would", "that'd've": "that would have", "that's": "that is",
  "there'd": "there had", "there'd've": "there would have", "there's": "there is",
  "they'd": "they would", "they'd've": "they would have", "they'll": "they will",
  "they'll've": "they will have", "they're": "they are", "they've": "they have",
  "to've": "to have", "wasn't": "was not", "we'd": "we had", "we'd've": "we would have",
  "we'll": "we will", "we'll've": "we will have", "we're": "we are", "we've": "we have",
  "weren't": "were not", "what'll": "what will", "what'll've": "what will have",
  "what're": "what are", "what's": "what is", "what've": "what have", "when's": "when is",
  "when've": "when have", "where'd": "where did", "where's": "where is",
  "where've": "where have", "who'll": "who will", "who'll've": "who will have", "who's": "who is",
  "who've": "who have", "why's": "why is", "why've": "why have", "will've": "will have",
  "won't": "will not", "won't've": "will not have", "would've": "would have",
  "wouldn't": "would not", "wouldn't've": "would not have", "y'all": "you all",
  "y'alls": "you alls", "y'all'd": "you all would", "y'all'd've": "you all would have",
  "y'all're": "you all are", "y'all've": "you all have", "you'd": "you had",
  "you'd've": "you would have", "you'll": "you you will", "you'll've": "you you will have",
  "you're": "you are", "you've": "you have"
}

def normalize_text(text):
    """
    入力テキストのノイズ除去および短縮形の展開を行うテキスト正規化関数。
    NLPモデル（Transformer等）の入力として最適なクリーンな文字列を生成する。
    """
    if text is None: return ""
    text = str(text)

    # 1. 英数字および特定の約物（Punctuation）以外の不要な文字を除去
    text = re.sub(r'[^a-zA-Z0-9\s\'.,?!#]:', '', text)

    # 2. 連続する同一の句読点（例: '!!!' -> '!'）を単一文字に圧縮（正規化）
    text = re.sub(r'([.,?!])\1+', r'\1', text)

    # 3. 連続する改行や空白文字を単一のスペースや改行に正規化し、両端をトリム
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 4. 正規表現を用いた字句解析（テキストを単語や記号のトークン単位に分割）
    words = re.findall(r"#?\b\w+[']?\w*\b|[^\s\w]+", text)
    expanded_words = []

    # 5. 各トークンに対して短縮形の展開処理を適用
    for word in words:
        lower_word = word.lower()
        if lower_word in cList:
            # 事前定義されたマッピング辞書(cList)を用いて短縮形を展開（Expansion）
            expanded = cList[lower_word]

            # 元トークンのキャピタライゼーション（先頭大文字）状態を保持して適用
            if word[0].isupper():
                expanded = expanded.capitalize()
            expanded_words.append(expanded)
        else:
            # 辞書に存在しない場合は元のトークンをそのまま保持
            expanded_words.append(word)

    # 6. 展開・正規化済みのトークン群を単一のスペース区切り文字列として再結合
    return " ".join(expanded_words)

## 4. Feature Engineering 
Kaggleのデータセットに含まれる `keyword` や `location` といったメタデータを、自然言語の本文（text）とプレフィックス付きで結合し、単一の入力シーケンスとして構築します。
これにより、DeBERTaのSelf-Attention機構が「メタデータ」と「本文」の相関関係を直接的に学習できるようになり、ノイズの多い短文分類における予測精度の大幅な向上が期待できます。

In [ ]:
def combine_features(row):
    """
    データフレームの各行（row）に対し、メタデータ（keyword, location）と
    本文（text）をプレフィックス付きで結合し、単一のテキストシーケンスを生成する。
    Transformerモデルにメタデータの文脈を付与するための前処理。
    """
    parts = []

    # 1. 欠損値（NaN）を判定し、有効な場合は'Keyword:'プレフィックスと共にリストへ追加
    if pd.notna(row['keyword']):
        parts.append(f"Keyword: {row['keyword']}")

    # 2. 欠損値（NaN）を判定し、有効な場合は'Location:'プレフィックスと共にリストへ追加
    if pd.notna(row['location']):
        parts.append(f"Location: {row['location']}")
    
    # 3. メインの入力特徴量である本文（text）を文字列キャストして追加
    parts.append(str(row['text']))

    # 4. 各構成要素をピリオドで結合し、単一のシーケンスとして返却
    return " . ".join(parts)

## 5. Token Sequence Validation
Transformerモデル（DeBERTa）の入力長制限（最大512トークン）に対する安全性を確認するため、テキストの正規化とトークン数の算出を同時に行う前処理パイプラインを定義します。

In [ ]:
def divide(df_workbook, name):
    """
    データフレーム内のテキストを正規化し、Transformerモデルに入力するための
    トークン数を算出・検証するデータ準備関数。
    """
    # 1. 特徴量統合とテキスト正規化パイプラインの適用
    df_workbook['combined_text'] = df_workbook.apply(combine_features, axis=1)
    df_workbook['cleaned_text'] = df_workbook['combined_text'].apply(normalize_text)

    # 2. トークナイザーによるエンコードとシーケンス長の算出
    df_workbook['token_len'] = df_workbook['cleaned_text'].apply(lambda x: len(tokenizer.encode(x, add_special_tokens=True)))

    # 3. 制限超過データのカウントと状態出力（ロギング）
    over_limit_count = len(df_workbook[df_workbook['token_len'] >= 512])
    print(f"512トークンを超えたデータ数: {over_limit_count} 件")

    return df_workbook

## 6. Execute Validation Pipeline
定義した検証関数を用いて、トレーニングデータおよびテストデータの前処理を一括で実行します。

In [ ]:
# 前処理およびシーケンス検証パイプラインの実行
df_test = divide(df_test, "test")
df_train = divide(df_train, "train")

## 7. Model Architecture & Hyperparameter Tuning
PyTorchを用いて、DeBERTaの最終隠れ層の上に分類タスク用のHeadを構築します。

In [ ]:
# 1. DeBERTaベースのカスタム分類器の定義
class BertClassifier(nn.Module):
    def __init__(self, model_name=bert_model):
        """
        事前学習済みモデルをBackboneとして読み込み、分類用Headを追加する。
        """
        super(BertClassifier, self).__init__()
        # 1. 事前学習済みDeBERTaモデル（特徴抽出器）の初期化
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # 2. 二値分類のための全結合層
        self.cls_layer = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        """
        順伝播の定義。
        Mean Poolingを用いてシーケンス全体の特徴ベクトルを抽出する。
        """
        # Transformer層へ入力し、各トークンの隠れ状態を取得
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        # Mean Pooling処理
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask 

        # 分類Headに平均ベクトルを入力し、ロジットを出力    
        logits = self.cls_layer(mean_embeddings)
        return logits

# 2. 学習準備（データ分割と最適化関数の設定）
# トレーニングデータから評価用データを分割 
df_train_sub, df_val = train_test_split(df_train, test_size=0.2, random_state=42)

# ファインチューニング用のオプティマイザ設定
optimizer = AdamW(model.parameters(), lr=2e-5)

# 二値分類タスク用の損失関数
criterion = nn.CrossEntropyLoss()

## 7. Model Evaluation & Validation
ホールドアウト検証データセットを用いて、モデルの汎化性能を評価する関数を定義します。
本実装では標準的な `DataLoader` を使用せず、DataFrameからの動的ミニバッチ生成と On-the-fly でのトークナイズ処理を組み合わせることで、推論時のメモリ効率と処理速度のバランスを最適化しています。

In [ ]:
from sklearn.metrics import accuracy_score

# 評価・検証関数の定義
def evaluate(df, target_model):
    """
    ホールドアウト検証データを用いてモデルの分類性能を評価する関数。
    DataFrameから動的にミニバッチを生成し、推論を実行する。
    """
    # 1. モデルを評価モードに切り替え
    target_model.eval()

    all_preds = []
    all_labels = []

    # ハードウェアのVRAM容量を考慮した推論用バッチサイズの設定
    batch_size = 8

    # 2. 勾配計算の無効化
    with torch.no_grad():

        # 3. 動的ミニバッチ生成ループ
        for i in range(0, len(df), batch_size):
            # バッチサイズ分だけDataFrameをスライスして抽出
            batch_df = df.iloc[i : i + batch_size]

            # 4. On-the-flyでのトークナイズ処理とテンソル化
            inputs = tokenizer.batch_encode_plus(
                batch_df['cleaned_text'].tolist(), 
                padding=True, truncation=True, max_length=128, return_tensors='pt'
            ).to(device)

            # 順伝播の実行と予測スコアの取得
            outputs = target_model(inputs['input_ids'], inputs['attention_mask'])

            # 6. 予測ラベルの抽出と評価指標計算のためのCPU・NumPy転送
            # 最も確率の高いクラスを取得
            preds = torch.argmax(outputs, dim=1)

            # scikit-learnの関数で処理するため、GPU上のテンソルをCPUメモリへ移動しNumPy配列に変換
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_df['target'].values)

    # 7. 全バッチの予測結果を統合し、最終的なAccuracyを算出
    return accuracy_score(all_labels, all_preds)

## 8. Hyperparameter Optimization
Optunaを活用して、モデルの性能を最大化するハイパーパラメータの探索空間を自動探索します。
本パイプラインでは、Transformerの微調整におけるベストプラクティスである「ウォームアップ付き線形学習率スケジューラ」を採用し、学習初期の不安定な勾配による重みの破壊を防ぎます。また、各TrialごとにEarly StoppingとVRAMのガベージコレクションを行うことで、過学習とメモリ枯渇を同時に防ぐ堅牢な設計としています。

In [ ]:
import optuna

# Optuna 目的関数（Objective Function）の定義
def objective(trial):
    """
    Optunaによるハイパーパラメータ探索のための目的関数。
    動的バッチングを用いた学習ループとEarly Stoppingによる評価を行う。
    """
    # 1. 探索するハイパーパラメータ空間の定義
    # 学習率は対数スケールで探索し、微小な変動を効率的に評価する
    lr = trial.suggest_float("lr", 1e-6, 5e-5, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16])

    # 最大エポック数の設定
    epochs = 5 
    
    # 2. モデルとオプティマイザの初期化
    model = BertClassifier().to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    
    # 3. 学習率スケジューラの設定
    # 全学習ステップ数を算出し、最初の10%をウォームアップ期間とする
    num_train_steps = int(len(df_train_sub) / batch_size * epochs)
    num_warmup_steps = int(num_train_steps * 0.1)

    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=num_warmup_steps, 
        num_training_steps=num_train_steps
    )

    # 二値分類用損失関数
    criterion = nn.CrossEntropyLoss()

    # 4. Early Stoppingの制御変数の初期化
    best_val_acc = 0.0
    patience = 1 
    counter = 0

    # 5. メイン学習ループ
    for epoch in range(epochs):
        # 5.1. Training Phase
        model.train()

        for i in range(0, len(df_train_sub), batch_size):
            # 勾配の初期化
            optimizer.zero_grad()

            # 動的ミニバッチ生成とOn-the-flyトークナイズ
            batch_df = df_train_sub.iloc[i : i + batch_size]
            inputs = tokenizer.batch_encode_plus(
                batch_df['cleaned_text'].tolist(), 
                padding=True, truncation=True, max_length=128, return_tensors='pt'
            ).to(device)
            labels = torch.tensor(batch_df['target'].values).to(device)

            # 順伝播と損失計算
            outputs = model(inputs['input_ids'], inputs['attention_mask'])
            loss = criterion(outputs, labels)

            # 逆伝播による勾配計算と重みの更新
            loss.backward()
            optimizer.step()

            # スケジューラによる学習率の減衰
            scheduler.step()

        # 5.2. Validation Phase
        val_acc = evaluate(df_val, model)
        
        # 5.3. Early Stoppingの判定
        if val_acc > best_val_acc:
            # 精度が改善した場合、ベストスコアを更新しカウンターをリセット
            best_val_acc = val_acc
            counter = 0
        else:
            # 精度が改善しなかった場合、カウンターをインクリメント
            counter += 1
            
        # 許容回数に達した場合、学習を早期に打ち切る
        if counter >= patience:
            break

    # 6. リソース解放
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
            
    return best_val_acc

## 9. Execute Hyperparameter Optimization
定義した探索空間と目的関数に基づき、Optunaによるハイパーパラメータの自動探索を実行します。
本パイプラインでは評価指標としてAccuracyを採用しているため、スコアの最大化を目的としたStudyセッションを構築します。

In [ ]:
# Optuna Studyの初期化と最適化プロセスの実行
# 1. Studyオブジェクトの作成
study = optuna.create_study(direction="maximize")

# 2. ハイパーパラメータ探索の実行
study.optimize(objective, n_trials=10)

# 3. 探索結果の評価とベストパラメータの抽出
print("一番良かった正解率:", study.best_value)
print("その時のパラメータ:", study.best_params)

## 10. Final Model Training
OptunaのHPOによって特定された最適なハイパーパラメータを利用し、最終的な推論用モデルを構築します。
ここでは、ホールドアウト検証のために分割していたデータセットを統合し、利用可能な全トレーニングデータを用いて再学習を行います。これにより、モデルが学習できる情報量を最大化し、未知のテストデータに対する汎化性能を極限まで引き上げます。

In [ ]:
# 最終モデルの構築と全データを用いた再学習
# 1. OptunaのStudyから、特定された最適なハイパーパラメータを取得
best_params = study.best_params

# 2. 本番用の新しいモデルインスタンスを初期化し、計算デバイスへ転送
final_model = BertClassifier().to(device)

# 最適な学習率を適用したオプティマイザの定義
optimizer = AdamW(final_model.parameters(), lr=best_params['lr'])

# モデルを学習モードへ切り替え
final_model.train()

# 3. 全トレーニングデータを用いたメイン学習ループ
# HPOフェーズでの収束傾向を考慮し、過学習を回避するため固定の3エポックで学習
for epoch in range(3):

    # 探索された最適なバッチサイズを用いて動的ミニバッチを生成
    for i in range(0, len(df_train), best_params['batch_size']):

        # 勾配の初期化
        optimizer.zero_grad()

        # バッチサイズ分のデータをDataFrameから抽出
        batch_df = df_train.iloc[i : i + best_params['batch_size']]

        # On-the-flyでのトークナイズ処理
        inputs = tokenizer.batch_encode_plus(
            batch_df['cleaned_text'].tolist(), 
            padding=True, truncation=True, max_length=128, return_tensors='pt'
        ).to(device)

        # 正解ラベルのテンソル化とGPU転送
        labels = torch.tensor(batch_df['target'].values).to(device)

        # 順伝播の実行と予測スコアの取得
        outputs = final_model(inputs['input_ids'], inputs['attention_mask'])
        
        # 損失の計算
        loss = nn.CrossEntropyLoss()(outputs, labels)

        # 逆伝播による誤差逆伝播と勾配計算
        loss.backward()

        # 最適化アルゴリズムによる重みパラメータの更新
        optimizer.step()

## 11. Inference & Submission
再学習した最終モデルを用いて、未知のテストデータに対する推論を実行します。
推論フェーズでは勾配情報の保持が不要なため、学習時よりもバッチサイズを拡大して処理速度を最適化しています。また、独自のPyTorchモデルとHugging Face標準モデルの双方に適合する堅牢なロジックを採用し、予測結果をKaggleの規定フォーマットに従ってCSVファイルに出力します。

In [ ]:
# 1. 推論（Inference）関数の定義
def predict(df, target_model):
    """
    未知のテストデータに対してバッチ推論を行い、予測ラベルのリストを生成する関数。
    """
    # 1.1. モデルを評価モードに切り替え
    target_model.eval()
    batch_size = 32
    all_predictions = []
    
    # 1.2. 勾配計算の完全無効化
    with torch.no_grad():
        # tqdmを用いて推論の進捗状況を可視化
        for i in tqdm(range(0, len(df), batch_size), desc="Predicting"):
            
            # 動的ミニバッチの抽出とOn-the-flyトークナイズ
            batch_df = df.iloc[i : i + batch_size]
            texts = batch_df['cleaned_text'].tolist()
            
            inputs = tokenizer.batch_encode_plus(
                texts, padding=True, truncation=True, 
                max_length=128, return_tensors='pt'
            ).to(device)
            
            # 1.3. 順伝播の実行
            outputs = target_model(inputs['input_ids'], inputs['attention_mask'])

            # 1.4. 出力形式の差異を吸収する堅牢なロジット抽出
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs
            
            # 1.5. 予測ラベルの確定
            preds = torch.argmax(logits, dim=1)

            # CPUメモリへ転送しNumPy配列に変換した上で、全体リストへ追加
            all_predictions.extend(preds.cpu().numpy())
            
    return all_predictions

# 2. 推論の実行と提出用（Submission）ファイルの作成
# 2.1. テストデータセットに対して最終モデルで推論を実行
test_predictions = predict(df_test, final_model)

# 2.2. Kaggleの提出規定に準拠したDataFrameの構築
submission = pd.DataFrame({
    "id": test_ids,
    "target": test_predictions
})

# 2.3. CSVファイルへのエクスポート
submission.to_csv("submission.csv", index=False)